In [1]:
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, r2_score, classification_report,
                             confusion_matrix, roc_auc_score)
try:
    from xgboost import XGBRegressor, XGBClassifier
    BACKEND = "XGBoost"
except ImportError:  # fallback only so the script still runs where xgboost is missing
    from sklearn.ensemble import GradientBoostingRegressor as XGBRegressor
    from sklearn.ensemble import GradientBoostingClassifier as XGBClassifier
    BACKEND = "sklearn GradientBoosting (xgboost not installed)"
 
OUT = ""   # charts saved next to the script
sns.set_theme(style="whitegrid")

In [2]:
#1. LOAD
Dataset_1 = pd.read_csv("Maharashtra_Electricity_Distribution_cleaned.csv")
Dataset_2 = pd.read_csv("POWER_Point_Daily_2025_cleaned.csv")
w = Dataset_2.copy()
w["date"] = pd.to_datetime(w["date"])
w = w.rename(columns={"solar_irradiance_kwh_m2_day": "irr", "temperature_2m_c": "temp"})
m = Dataset_1.rename(columns={"Installed Capacity in MW": "install_mw",
                      "Solar Plant Capacity in MW": "solar_mw",
                      "Land Required in Acre": "land_acre"})
 
w["month"] = w.date.dt.month
w["doy"] = w.date.dt.dayofyear
 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
 
CAP_KW = 10.0
def make_xgb_clf():
    if BACKEND == "XGBoost":      # small + heavily regularised: the key to higher accuracy here
        return XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, subsample=0.8,
                             min_child_weight=5, reg_lambda=5, random_state=0)
    return XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, subsample=0.8,
                         min_samples_leaf=10, random_state=0)
 
def simulate(seed):
    rng = np.random.default_rng(seed)
    d = w.copy()
    tcell = d.temp + 4.0 * d.irr
    d["expected_kwh"] = CAP_KW * d.irr * 0.80 * (1 - 0.004 * (tcell - 25))
    rain = (d.irr < 3.0).to_numpy(); soil = np.zeros(len(d)); s = 0.0
    for i in range(len(d)):
        s = 0.0 if rain[i] else min(s + 0.002, 0.15); soil[i] = s
    shade = np.zeros(len(d))
    for st in rng.choice(len(d) - 12, 6, replace=False):
        shade[st:st + rng.integers(4, 12)] = rng.uniform(0.08, 0.18)
    equip = np.zeros(len(d))
    for st in rng.choice(len(d) - 25, 3, replace=False):
        equip[st:st + rng.integers(7, 25)] = rng.uniform(0.20, 0.45)
    loss = 1 - (1 - soil) * (1 - shade) * (1 - equip)
    d["true_loss"] = loss
    d["actual_kwh"] = (d.expected_kwh * (1 - loss) * rng.normal(1, 0.02, len(d))).clip(lower=0)
    d["fault"] = np.select([equip > 0, shade > 0, soil > 0.08], ["equipment", "shading", "soiling"], "healthy")
    d["needs_inspection"] = (loss > 0.10).astype(int)
    return d
 
def evaluate(seed, verbose=False):
    d = simulate(seed)
    # (A) richer healthy-baseline: weather + cloud persistence + season
    d["irr_lag1"] = d.irr.shift(1).bfill(); d["irr_3d"] = d.irr.rolling(3, min_periods=1).mean()
    d["doy_sin"] = np.sin(2 * np.pi * d.doy / 365); d["doy_cos"] = np.cos(2 * np.pi * d.doy / 365)
    f1 = ["irr", "temp", "irr_lag1", "irr_3d", "doy_sin", "doy_cos"]
    tr = d[d.fault == "healthy"].sample(frac=0.7, random_state=1).index
    reg = XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.04, random_state=0)
    reg.fit(d.loc[tr, f1], d.loc[tr, "actual_kwh"])
    d["pred_kwh"] = reg.predict(d[f1])
    # (B) FEW smoothed features: actual/expected now, 3-day and 7-day averages
    d["ratio"] = d.actual_kwh / d.pred_kwh.clip(lower=0.5)
    d["m3"] = d.ratio.rolling(3, min_periods=1).mean(); d["m7"] = d.ratio.rolling(7, min_periods=1).mean()
    feats = ["ratio", "m3", "m7"]
    cut = int(len(d) * 0.7)                       # time-ordered split: train on past, test on future
    X, y = d[feats], d.needs_inspection
    clf = make_xgb_clf().fit(X[:cut], y[:cut])    # (C) small, regularised boosted trees
    pred = clf.predict(X[cut:]); rule = (d.ratio[cut:] < 0.9).astype(int)
    if verbose:
        prob = clf.predict_proba(X[cut:])[:, 1]
        print("[Baseline v2] MAE=%.2f kWh" % mean_absolute_error(d.loc[d.index.difference(tr), "actual_kwh"][d.fault=="healthy"],
              d.loc[d.index.difference(tr), "pred_kwh"][d.fault=="healthy"]))
        print(classification_report(y[cut:], pred, target_names=["ok", "inspect"], digits=2))
        print("ROC-AUC = %.3f" % roc_auc_score(y[cut:], prob))
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
        sns.heatmap(confusion_matrix(y[cut:], pred), annot=True, fmt="d", cmap="Blues",
                    xticklabels=["ok", "inspect"], yticklabels=["ok", "inspect"], ax=ax[0]); ax[0].set_title("Improved model (test)")
        pd.Series(clf.feature_importances_, index=feats).sort_values().plot.barh(ax=ax[1], color="tab:green")
        ax[1].set_title("Feature importance"); plt.tight_layout(); plt.savefig(OUT + "05_improved_model.png", dpi=130); plt.close()
    return accuracy_score(y[cut:], pred), accuracy_score(y[cut:], rule)
 
print("=== Single run (seed 42) ===")
a, r_ = evaluate(42, verbose=True)
print("Accuracy: improved model %.3f | simple rule 'ratio<0.90' %.3f | previous model 0.918" % (a, r_))
 
print("\n=== Robust benchmark: 20 different simulated years ===")
res = np.array([evaluate(s) for s in range(20)])
print("Improved model: mean acc %.3f (std %.3f)" % (res[:, 0].mean(), res[:, 0].std()))
print("Simple rule   : mean acc %.3f (std %.3f)" % (res[:, 1].mean(), res[:, 1].std()))
print("Model beats rule in %d of 20 simulations" % (res[:, 0] > res[:, 1]).sum())
print("Backend used:", BACKEND)

=== Single run (seed 42) ===
[Baseline v2] MAE=0.97 kWh
              precision    recall  f1-score   support

          ok       0.91      0.99      0.95        85
     inspect       0.94      0.68      0.79        25

    accuracy                           0.92       110
   macro avg       0.93      0.83      0.87       110
weighted avg       0.92      0.92      0.91       110

ROC-AUC = 0.967
Accuracy: improved model 0.918 | simple rule 'ratio<0.90' 0.918 | previous model 0.918

=== Robust benchmark: 20 different simulated years ===
Improved model: mean acc 0.945 (std 0.034)
Simple rule   : mean acc 0.905 (std 0.034)
Model beats rule in 17 of 20 simulations
Backend used: XGBoost
